# PostgreSQL queries — 2NF before vs after

Use this notebook to explore why 2NF matters: product attributes repeated on every
line item cause update anomalies; after normalization, one product row is enough.

Uses the shared `scripts.db` helpers (same connection as `make seed` / `seed.ipynb`).

Run `make compose_up` and seed via `seed.ipynb` or `make seed`, then open this notebook at `http://localhost:8888`.

In [ ]:
from scripts.db import query

## 1NF table (violates 2NF)

Partial dependency: `product_name` / `unit_price_cents` depend only on `product_sku`.

In [ ]:
query(
    """
    SELECT order_id, product_sku, product_name, unit_price_cents, quantity
    FROM order_items_1nf
    ORDER BY product_sku, order_id
    LIMIT 15
    """
)

### Redundancy: same SKU, repeated product attributes

Count how many times each SKU's name/price is duplicated across order lines.

In [ ]:
query(
    """
    SELECT
      product_sku,
      product_name,
      unit_price_cents,
      COUNT(*) AS line_count
    FROM order_items_1nf
    GROUP BY product_sku, product_name, unit_price_cents
    ORDER BY line_count DESC
    """
)

### Pain point: update anomaly

Renaming a product requires updating **every** line that uses that SKU.
The next cell shows how many rows would change for the most common SKU.

In [ ]:
query(
    """
    SELECT product_sku, COUNT(*) AS rows_to_update
    FROM order_items_1nf
    WHERE product_sku = (
      SELECT product_sku
      FROM order_items_1nf
      GROUP BY product_sku
      ORDER BY COUNT(*) DESC
      LIMIT 1
    )
    GROUP BY product_sku
    """
)

## 2NF tables

Product attributes live once in `products`; `order_items` keeps only `quantity`.

In [ ]:
query(
    """
    SELECT sku, name, unit_price_cents
    FROM products
    ORDER BY sku
    """
)

In [ ]:
query(
    """
    SELECT order_id, product_sku, quantity
    FROM order_items
    ORDER BY order_id, product_sku
    LIMIT 15
    """
)

### Same question, cleaner answer: rename updates one row

After 2NF, changing a product name touches exactly one row in `products`.

In [ ]:
query(
    """
    UPDATE products
    SET name = name || ' (renamed)'
    WHERE sku = (SELECT sku FROM products ORDER BY sku LIMIT 1)
    RETURNING sku, name, unit_price_cents
    """
)

### Join for order totals

Price comes from `products`; quantity from `order_items`.

In [ ]:
query(
    """
    SELECT
      oi.order_id,
      SUM(oi.quantity * p.unit_price_cents) AS total_cents
    FROM order_items oi
    JOIN products p ON p.sku = oi.product_sku
    GROUP BY oi.order_id
    ORDER BY oi.order_id
    LIMIT 20
    """
)

## Row counts (sanity check)

Line-item count stays the same; product count equals distinct SKUs from the 1NF table
(far fewer than line items — redundancy removed).

In [ ]:
query(
    """
    SELECT
      (SELECT COUNT(*) FROM order_items_1nf) AS order_items_1nf,
      (SELECT COUNT(DISTINCT product_sku) FROM order_items_1nf) AS distinct_skus_1nf,
      (SELECT COUNT(*) FROM products) AS products,
      (SELECT COUNT(*) FROM order_items) AS order_items_2nf
    """
)